In [0]:
# Notebook : raw_to_landing_hr
# Purpose : Ingestion of HR Domain files from ADLS to UC Volume
# Domain : HR
# Files : HR.csv
# Author : Virendra Dilip Tambavekar
# HRMID : 6217
# Created Date : 14/05/2026

In [0]:
# importing logging and sql functions, types
import logging
import uuid
from pyspark.sql.functions import *
from pyspark.sql.types import StructType, StructField, StringType

# initializing logger
logger = logging.getLogger("RawToLandingHR")
logger.setLevel(logging.INFO)

In [0]:
# Setting Widgets : Catalog, Batch ID, run_id
dbutils.widgets.text("catalog","charles_schwab_retailbrokerage_dev_team_lemma")
dbutils.widgets.text("batch_id","1")

catalog = dbutils.widgets.get("catalog")
batch_id = dbutils.widgets.get("batch_id")

#Generating Unique run_id
run_id = str(uuid.uuid4())

#Paths
source_file_path = f"abfss://raw@schwabdldevsa.dfs.core.windows.net/Batch{batch_id}/HR.csv"
target_landing_path = f"/Volumes/charles_schwab_retailbrokerage_dev_team_lemma/landing/pwg/Batch{batch_id}/hr/"

In [0]:
#Defining Explicit Schema to prevent Data Loss
hr_schema = StructType([
    StructField("EMPLOYEE_ID",StringType(),True),
    StructField("MANAGER_ID",StringType(), True),
    StructField("LAST_NAME",StringType(),True),
    StructField("FIRST_NAME",StringType(),True),
    StructField("MIDDLE_INITIAL",StringType(),True),
    StructField("JOB_CODE",StringType(),True),
    StructField("BRANCH_ID",StringType(),True),
    StructField("OFFICE",StringType(),True),
    StructField("PHONE",StringType(),True)
])

In [0]:
# Idempotency Logic

def idempotency_check_landing(target_path):
    #Check if landing path already has data and log state before overwrite.
    try:
        files = dbutils.fs.ls(target_path)
        if files:
            logger.info(f"Idempotency: Target path '{target_path}' already has {len(files)} file(s). Will be overwritten.")
        else:
            logger.info(f"Idempotency: Target path '{target_path}' is empty. Will write fresh.")
    except Exception:
        logger.info(f"Idempotency: Target path '{target_path}' does not exist. Will be created.")

idempotency_check_landing(target_landing_path)

In [0]:
def ingest_hr():
    logger.info(f"Raw to landing ingestion for HR.csv in Batch{batch_id}. Run ID:{run_id}")
    #Exception Handling for HR.csv
    try:
        dbutils.fs.ls(source_file_path)
    except Exception:
        logger.info(f"HR.csv not found in Batch{batch_id}")
        return False
    #Read CSV
    raw_df = spark.read.csv(source_file_path, header=False, schema=hr_schema)

    #Adding Metadata
    landing_df = (
        raw_df
            .withColumn("_landing_ts", current_timestamp())
            .withColumn("_batch_id", lit(batch_id))
            .withColumn("_run_id", lit(run_id))
            .withColumn("_source_file", lit("HR.csv"))
    )

    #Writing Parquet file to UC in Parquet format.
    landing_df.write.mode("overwrite").parquet(target_landing_path)
    logger.info(f"Ingested HR.csv to {target_landing_path}")
    return True

file_processed = ingest_hr()

In [0]:
if file_processed:
    try:
        landing_df = spark.read.parquet(target_landing_path)
        landing_count = landing_df.count()
        logger.info(f"Landing count for HR.csv : {landing_count}")
        display(landing_df.limit(10))
    except Exception as e:
        logger.error(f"Validation failed for HR.csv : {str(e)}")
        raise e